# 01 Baseline and Feature Comparison

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`c3.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/scripts/c3.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/logs)
- [`run_all.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/scripts/run_all.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/HGB/Combined/scripts/c3.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '8a8f5067909154cbd6af5a8dd3f7f35e952dce96'}, '05_model_optimize/HGB/Combined/scripts/run_all.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'bd3e0f3e2d677fe09c2b8b882091c915336b5daa'}}

## c3

원본: [05_model_optimize/HGB/Combined/scripts/c3.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/scripts/c3.py)  
Git blob: `8a8f5067909154cbd6af5a8dd3f7f35e952dce96`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Combined/scripts/c3.py
from pathlib import Path
import sys,time,json,numpy as np
w=int(sys.argv[1]); sys.argv=['x','none']; exec(open(Path(__file__).resolve().parent/'run_all.py').read().split("stage=sys.argv[1]")[0])
Ftr=np.load(OUT/'Ftr.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}
keep=np.r_[0:784,840:1164]
o=[x for x in W if x!=w]; X=np.vstack([Ftr[:,keep]]+[Fm[x][:,keep] for x in o]); y=np.concatenate([ytr]+[wr[x]['y'] for x in o])
m=hgb(); t=time.time(); m.fit(X,y); fs=time.time()-t; p=m.predict(Fm[w][:,keep])
np.save(OUT/f'c3_{w}.npy',p); json.dump(dict(writer=w,acc=accuracy_score(wr[w]['y'],p),fit_s=round(fs),iters=int(m.n_iter_)),open(OUT/f'c3_{w}.json','w')); print(w,accuracy_score(wr[w]['y'],p),m.n_iter_,flush=True)


## run_all

원본: [05_model_optimize/HGB/Combined/scripts/run_all.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/scripts/run_all.py)  
Git blob: `bd3e0f3e2d677fe09c2b8b882091c915336b5daa`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/HGB/Combined/logs)

In [ ]:
%%archive_source 05_model_optimize/HGB/Combined/scripts/run_all.py
from pathlib import Path
import sys, time, json, numpy as np, pandas as pd
from pathlib import Path
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
REPO=Path(__file__).resolve().parents[4]; OUT=Path(__file__).resolve().parents[2]/'Normal'/'scripts'/'out'; OUT.mkdir(exist_ok=True)
sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Combined/scripts')); sys.path.insert(0,str(REPO/'05_model_optimize/MLP/Normal/scripts'))
from combined_mlp_writer_cv import load_official, load_writers
from exp006_size_com_comparison import process
import exp_cmb005_balanced_scale_stroke as L
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,ConfusionMatrixDisplay
from skimage.feature import hog
import sklearn; print('sklearn',sklearn.__version__,flush=True)
off=load_official(REPO/'02_data/datasets/MNIST_EMNIST_official_split_112000.npz'); wr=load_writers(REPO/'02_data/Hand_made')
W=list(L.WRITERS); LAB='0123456789ABCDEF'; ytr,yva=off['y_train'],off['y_val']
yhm=np.concatenate([wr[w]['y'] for w in W]); whm=np.concatenate([[w]*640 for w in W])
Xtr,tw,_,_=L.prepareraw(off,wr,'scale30_stroke28'); Xva=process(off['X_val'],'com',20); Xhm=np.vstack([tw[w] for w in W])
PAIRS=[('D','0'),('0','D'),('B','8'),('8','B'),('5','F'),('7','9')]
def hgb(**k): return HistGradientBoostingClassifier(max_iter=300,early_stopping=True,validation_fraction=0.1,n_iter_no_change=10,random_state=42,**k)
def sc(y,p,pre): return {f'{pre}_acc':accuracy_score(y,p),f'{pre}_precision':precision_score(y,p,average='macro',zero_division=0),f'{pre}_recall':recall_score(y,p,average='macro',zero_division=0),f'{pre}_f1':f1_score(y,p,average='macro')}
def det(p):
    d=yhm<10; r=dict(hm_digit=accuracy_score(yhm[d],p[d]),hm_letter=accuracy_score(yhm[~d],p[~d]))
    for w in W: r[f'w{w}']=accuracy_score(yhm[whm==w],p[whm==w])
    for a,b in PAIRS: r[f'{a}->{b}']=int(np.sum((yhm==LAB.index(a))&(p==LAB.index(b))))
    return r
res=[]
def save(r): res.append(r); pd.DataFrame(res).to_csv(OUT/'results.csv',index=False); print(r,flush=True)
stage=sys.argv[1]
if stage=='n3':
    m=hgb(); t=time.time(); m.fit(Xtr,ytr); fs=time.time()-t
    pv=m.predict(Xva); ph=m.predict(Xhm)
    save(dict(name='N3',**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=m.n_iter_))
    np.save(OUT/'ph3.npy',ph)
    def gm(cls=None):
        g=np.zeros(784)
        for it in m._predictors:
            for tr in (it if cls is None else [it[cls]]):
                n=tr.nodes[tr.nodes['is_leaf']==0]; np.add.at(g,n['feature_idx'],n['gain'])
        return g.reshape(28,28)
    maps={'All':gm()}|{c:gm(LAB.index(c)) for c in 'D0B8'}
    np.savez(OUT/'gain.npz',**maps)
    fig,ax=plt.subplots(1,5,figsize=(20,4.5))
    for a,(k,g) in zip(ax,maps.items()): a.imshow(g,cmap='hot'); a.set_title(f'{k} trees'); a.axis('off')
    plt.tight_layout(); plt.savefig(OUT/'D1_gain_heatmap.png',dpi=130)
    st={}
    for k,g in maps.items():
        s=np.sort(g.ravel())[::-1]; st[k]=dict(used=int((g>0).sum()),top20=float(s[:20].sum()/s.sum()))
    json.dump(st,open(OUT/'D1_stats.json','w')); print(st,flush=True)
if stage=='n5':
    def sf(X):
        X=X.reshape(-1,28,28).astype(np.float32); proj=np.hstack([X.sum(1),X.sum(2)])/255.
        return np.hstack([proj,np.stack([hog(i,orientations=9,pixels_per_cell=(7,7),cells_per_block=(2,2)) for i in X])]).astype(np.float32)
    t=time.time(); Ftr=np.hstack([Xtr,sf(Xtr)]); Fva=np.hstack([Xva,sf(Xva)]); Fm={w:np.hstack([tw[w],sf(tw[w])]) for w in W}; Fhm=np.vstack([Fm[w] for w in W]); fe=time.time()-t
    np.save(OUT/'Ftr.npy',Ftr); np.save(OUT/'Fva.npy',Fva); np.savez(OUT/'Fm.npz',**{str(k):v for k,v in Fm.items()})
    m=hgb(); t=time.time(); m.fit(Ftr,ytr); fs=time.time()-t
    t=time.time(); pv=m.predict(Fva); pvs=time.time()-t; t=time.time(); ph=m.predict(Fhm); phs=time.time()-t
    t=time.time(); m.predict(Fva[:1]); one=time.time()-t
    save(dict(name='N5',**sc(yva,pv,'val'),**sc(yhm,ph,'hm'),**det(ph),fit_s=round(fs),iters=m.n_iter_,feature_s=round(fe,1),
              predict_val_s=round(pvs,3),predict_hm_s=round(phs,3),ms_per_img=round(1000*pvs/len(yva),4),single_img_ms=round(1000*one,2)))
    np.save(OUT/'ph5.npy',ph)
    it=np.arange(len(m.train_score_))
    av=[accuracy_score(yva,p) for p in m.staged_predict(Fva)]; ah=[accuracy_score(yhm,p) for p in m.staged_predict(Fhm)]
    lc=pd.DataFrame({'iteration':it,'train_loss':-m.train_score_,'val_loss':-m.validation_score_,'official_val_acc':[np.nan]+av[:len(it)-1],'handmade_acc':[np.nan]+ah[:len(it)-1]})
    lc.to_csv(OUT/'learning_curve.csv',index=False); print(lc.iloc[[1,10,50,100,-1]],flush=True)
if stage.startswith('c2_'):
    w=int(stage[3:]); Ftr=np.load(OUT/'Ftr.npy'); Fm={int(k):v for k,v in np.load(OUT/'Fm.npz').items()}
    o=[x for x in W if x!=w]; X=np.vstack([Ftr]+[Fm[x] for x in o]); y=np.concatenate([ytr]+[wr[x]['y'] for x in o])
    m=hgb(); t=time.time(); m.fit(X,y); fs=time.time()-t; t=time.time(); p=m.predict(Fm[w]); ps=time.time()-t
    np.save(OUT/f'c2_{w}.npy',p); json.dump(dict(writer=w,train_n=len(y),acc=accuracy_score(wr[w]['y'],p),fit_s=round(fs),iters=int(m.n_iter_),predict_s=round(ps,3)),open(OUT/f'c2_{w}.json','w'))
    print('fold',w,accuracy_score(wr[w]['y'],p),round(fs),m.n_iter_,flush=True)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")